In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import functions as F
import builtins

In [0]:
def silver_intake(path,label:str):
  """
  This function will take silver tables from silver schema
  """
  df=spark.table(path)
  print(f'{label} table is fetched successfully..')
  return df
gold_tb_1=silver_intake('dev_catalog.silver_schema.bank_silver_table','Bank')
gold_tb_2=silver_intake('dev_catalog.silver_schema.bigmartsales_silver_table','BigMartSales')

Bank table is fetched successfully..
BigMartSales table is fetched successfully..


In [0]:
gold_tb_1.write.format("delta").mode("overwrite").saveAsTable("dev_catalog.gold_schema.gold_tb_1")
gold_tb_2.write.format("delta").mode("overwrite").saveAsTable("dev_catalog.gold_schema.gold_tb_2")

In [0]:
def Basic_stat(tb,label:str):
    """
    This function will give Basic Statistics of the table
    """

    print(f'Basic Statistics of {label} table')
    tb.describe().display()

In [0]:
Basic_stat(gold_tb_1,'Bank')

Basic Statistics of Bank table


summary,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
count,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521,4521
mean,41.17009511170095,null,null,null,null,1422.6578190665782,null,null,null,15.915284229152842,null,263.96129174961294,2.793629727936297,39.766644547666445,0.5425790754257908,null,null
stddev,10.576210958711261,null,null,null,null,3009.638142467309,null,null,null,8.24766732722993,null,259.85663262468205,3.1098066601885743,100.12112444301664,1.6935623506071236,null,null
min,19,admin.,divorced,primary,no,-3313,no,no,cellular,1,apr,4,1,-1,0,failure,no
max,87,unknown,single,unknown,yes,71188,yes,yes,unknown,31,sep,3025,50,871,25,unknown,yes


In [0]:
Basic_stat(gold_tb_2,'BigMartSales')

Basic Statistics of BigMartSales table


summary,Outlet_Type,Item_Identifier,Item_Weight,Item_Fat_Content,Item_Visibility,Item_Type,Item_MRP,Outlet_Identifier,Outlet_Establishment_Year,Outlet_Size,Outlet_Location_Type,Item_Outlet_Sales
count,8523,8523,8523,8523,8523,8523,8523,8523,8523,8523,8523,8523
mean,null,null,12.813419570574444,null,0.06613202877895127,null,140.9927819781768,null,1997.8318667135984,null,null,2181.2889135750365
stddev,null,null,4.227240406467763,null,0.05159782232113495,null,62.2750665121905,null,8.371760408092655,null,null,1706.4996157338335
min,Grocery Store,DRA12,4.555,LF,0.0,Baking Goods,31.29,OUT010,1985,High,Tier 1,33.29
max,Supermarket Type3,NCZ54,21.35,reg,0.328390948,Starchy Foods,266.8884,OUT049,2009,Small,Tier 3,13086.9648


In [0]:
def row_count(tb1,tb2,label1:str,label2:str):
    row1=tb1.count()
    row2=tb2.count()
    status='Pass' if row1==row2 else 'Fail'
    return [('Row Count',label1,row1,label2,row2,status)]

In [0]:
def column_count(tb1,tb2,label1:str,label2:str):
    col1=len(tb1.columns)
    col2=len(tb2.columns)
    status='Pass' if col1==col2 else 'Fail'
    return [('Column Count',label1,col1,label2,col2,status)]

In [0]:

def Null_Count_percent(tb1, tb2, label1, label2):
    r1 = tb1.count()
    r2 = tb2.count()

    nulls_tb1 = tb1.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in tb1.columns
    ])
    values_1 = nulls_tb1.collect()[0].asDict().values()
    total_nulls_1 = builtins.sum(v for v in values_1 if v is not None)

    nulls_tb2 = tb2.select([
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in tb2.columns
    ])
    values_2 = nulls_tb2.collect()[0].asDict().values()
    total_nulls_2 = builtins.sum(v for v in values_2 if v is not None)

    total_cells_1 = r1 * len(tb1.columns)
    total_cells_2 = r2 * len(tb2.columns)

    null_pct_1 = (total_nulls_1 / total_cells_1) * 100 if total_cells_1 else 0
    null_pct_2 = (total_nulls_2 / total_cells_2) * 100 if total_cells_2 else 0

    status = "PASS" if builtins.abs(null_pct_1 - null_pct_2) < 5 else "WARN"

    return [("Null Percentage", label1, builtins.round(null_pct_1, 2), label2, builtins.round(null_pct_2, 2), status)]

In [0]:
def duplicate_check(tb2, tb1, label1, label2):

    dup_1 = tb1.count() - tb1.dropDuplicates().count()
    dup_2 = tb2.count() - tb2.dropDuplicates().count()

    status = "PASS" if dup_1 == 0 and dup_2 == 0 else "FAIL"

    return [("Duplicate Rows", label1, dup_1, label2, dup_2, status)]

In [0]:
def numeric_mean_check(tb1, tb2, label1, label2):
    num_cols_1 = [c for c, t in tb1.dtypes if t in ('int','double','float','bigint')]
    num_cols_2 = [c for c, t in tb2.dtypes if t in ('int','double','float','bigint')]
    if num_cols_1:
        mean_1 = tb1.select([F.mean(c) for c in num_cols_1]).groupBy().avg().first()[0]
    else:
        mean_1 = 0
    if num_cols_2:
        mean_2 = tb2.select([F.mean(c) for c in num_cols_2]).groupBy().avg().first()[0]
    else:
        mean_2 = 0
    return [("Avg Numeric Mean", label1, builtins.round(mean_1,2), label2, builtins.round(mean_2,2), "INFO")]

In [0]:
schema = StructType([
    StructField("Check",  StringType(), True),
    StructField("Table1", StringType(), True),
    StructField("Value1", DoubleType(), True),
    StructField("Table2", StringType(), True),
    StructField("Value2", DoubleType(), True),
    StructField("Status", StringType(), True),
])

result = []
result += row_count(gold_tb_1, gold_tb_2, 'Bank', 'BigMartSales')
result += column_count(gold_tb_1, gold_tb_2, 'Bank', 'BigMartSales')
result += Null_Count_percent(gold_tb_1, gold_tb_2, 'Bank', 'BigMartSales')
result += duplicate_check(gold_tb_1, gold_tb_2, 'Bank', 'BigMartSales')
result += numeric_mean_check(gold_tb_1, gold_tb_2, 'Bank', 'BigMartSales')

# Explicit float cast 
result = [(c, t1, float(v1), t2, float(v2), s) for c, t1, v1, t2, v2, s in result]

validation_report = spark.createDataFrame(result, schema=schema)
display(validation_report)

Check,Table1,Value1,Table2,Value2,Status
Row Count,Bank,4521.0,BigMartSales,8523.0,Fail
Column Count,Bank,17.0,BigMartSales,12.0,Fail
Null Percentage,Bank,0.0,BigMartSales,0.0,PASS
Duplicate Rows,Bank,0.0,BigMartSales,0.0,PASS
Avg Numeric Mean,Bank,41.17,BigMartSales,12.81,INFO


Saving Validation_report as gold table

In [0]:
(validation_report.write.mode("overwrite")
    .option("overwriteSchema", "true")
    .option("delta.autoOptimize.optimizeWrite", "true")
    .option("delta.autoOptimize.autoCompact", "true")
    .saveAsTable(f'dev_catalog.gold_schema.Validation_report_gold'))

In [0]:
def Gold_to_adls(tb, path,tb_name:str):
    tb.coalesce(1).write.format("csv") \
        .option("header", True) \
        .mode("overwrite") \
        .save(path)
    print(f'{tb_name} saved successfully at {path}')

In [0]:
numeric_cols_g_tb1 = {f.name for f in gold_tb_1.schema.fields if isinstance(f.dataType, (DoubleType, IntegerType, FloatType))}
numeric_cols_g_tb2 = {f.name for f in gold_tb_2.schema.fields if isinstance(f.dataType, (DoubleType, IntegerType, FloatType))}
all_numeric_cols = list(numeric_cols_g_tb1.union(numeric_cols_g_tb2))
print("All numeric columns:", all_numeric_cols)

All numeric columns: ['day', 'balance', 'Item_Weight', 'Item_MRP', 'duration', 'Item_Outlet_Sales', 'previous', 'Item_Visibility', 'pdays', 'age', 'Outlet_Establishment_Year', 'campaign']


In [0]:
def get_mean_stdDev(tb,col_name):
    stats = tb.select(
        mean(col_name).alias("mean"),
        stddev(col_name).alias("stddev")
    ).collect()[0]
    
    return stats["mean"], stats["stddev"]

In [0]:
def mean_diff_pct(mean1, mean2):
    if mean1 == 0 or mean1 is None or mean2 is None:
        return None
    return abs(mean1 - mean2) / mean1 * 100

In [0]:
def get_outlier_pct(tb, col_name):
    q1, q3 = tb.approxQuantile(col_name, [0.25, 0.75], 0.01)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    total = tb.count()
    outliers = tb.filter((col(col_name) < lower) | (col(col_name) > upper)).count()
    return (outliers / total) * 100 if total > 0 else 0

In [0]:
valid_data=[]
for col_name in all_numeric_cols:
    mean1, stddev1 = get_mean_stdDev(gold_tb_1, col_name) if col_name in numeric_cols_g_tb1 else (None, None)
    mean2, stddev2 = get_mean_stdDev(gold_tb_2, col_name) if col_name in numeric_cols_g_tb2 else (None, None)
    # Mean diff %
    if mean1 is None or mean2 is None or mean1 == 0:
        mean_diff = None
        status = "INFO"
    else:
        mean_diff = mean_diff_pct(mean1, mean2)
        
        if mean_diff < 10:
            status = "PASS"
        elif mean_diff < 30:
            status = "WARNING"
        else:
            status = "FAIL"
    
    outlier1 = get_outlier_pct(gold_tb_1, col_name) if col_name in numeric_cols_g_tb1 else None
    outlier2 = get_outlier_pct(gold_tb_2, col_name) if col_name in numeric_cols_g_tb2 else None
    
   # Mean Diff row
    valid_data.append((
        f"{col_name} - Mean Diff %",
        "Bank", builtins.round(mean1,2) if mean1 else None,
        "BigMartSales", builtins.round(mean2,2) if mean2 else None,
        status
    ))
    
    # Std Dev row
    valid_data.append((
        f"{col_name} - StdDev",
        "Bank", builtins.round(stddev1,2) if stddev1 else None,
        "BigMartSales", builtins.round(stddev2,2) if stddev2 else None,
        "INFO"
    ))
    
    # Outlier row
    valid_data.append((
        f"{col_name} - Outlier %",
        "Bank", builtins.round(outlier1,2) if outlier1 else None,
        "BigMartSales", builtins.round(outlier2,2) if outlier2 else None,
        "INFO"
    ))

In [0]:
columns = ["Check", "Table1", "Value1", "Table2", "Value2", "Status"]

validation_df_new = spark.createDataFrame(valid_data, columns)

final_validation_report = validation_report.union(validation_df_new)
display(final_validation_report)

Check,Table1,Value1,Table2,Value2,Status
Row Count,Bank,4521.0,BigMartSales,8523.0,Fail
Column Count,Bank,17.0,BigMartSales,12.0,Fail
Null Percentage,Bank,0.0,BigMartSales,0.0,PASS
Duplicate Rows,Bank,0.0,BigMartSales,0.0,PASS
Avg Numeric Mean,Bank,41.17,BigMartSales,12.81,INFO
day - Mean Diff %,Bank,15.92,BigMartSales,null,INFO
day - StdDev,Bank,8.25,BigMartSales,null,INFO
day - Outlier %,Bank,null,BigMartSales,null,INFO
balance - Mean Diff %,Bank,1422.66,BigMartSales,null,INFO
balance - StdDev,Bank,3009.64,BigMartSales,null,INFO


In [0]:
final_validation_report.write.mode("overwrite") \
    .option("overwriteSchema", "true") \
    .option("delta.autoOptimize.optimizeWrite", "true") \
    .option("delta.autoOptimize.autoCompact", "true") \
    .saveAsTable(f'dev_catalog.gold_schema.final_validation_report')

In [0]:
Gold_to_adls(final_validation_report,'abfss://project-cont@1destorageacc.dfs.core.windows.net/gold/final_validation_report','final_validation_report')

final_validation_report saved successfully at abfss://project-cont@1destorageacc.dfs.core.windows.net/gold/final_validation_report
